# Transformation - Flight data

### Transformation Checklist
1. [ ] Remove unnecessary time in `fl_date` column and make sure its `dd/mm/yyyy`.
2. [ ] Type correction across all columns
3. [ ] Drop unncessary columns
   1. `op_carrier_airline_id` - Redundant
   2. `tail_num` - Unnecessary
   3. `origin_airport_seq_id` - Redandant
   4. `origin_city_market_id` - Unnecessary
   5. `origin_state_fips` - Unnecessary
   6. `origin_state_nm` - Redandant
   7. `origin_wac` - Unnecessary
   8. `dest_airport_seq_id` - Redundant
   9. `dest_state_nm` - Redundant
   10. `dest_wac` - Unnecessary
   11. `cancelled` - Cancellation status is not relevant for the analysis
   12. `cancellation_code` - Unnecessary
   13. `diverted` - Diversion status is not relevant for the analysis
   14. `actual_elapsed_time` - Flight time is not relevant for this analysis
   15. `flights` - Extreamly low variance column, only single value accoross the entire dataset
4. [ ] Deal with missing values
5. [ ] Standardize all the date fields to use the same format 

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import os
import logging

In [2]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (SparkSession
    .builder
    .appName("FlightDataTransformation")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())

spark.sparkContext.setLogLevel("FATAL")
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/rwjlb/.ivy2.5.2/cache
The jars for the packages stored in: /home/rwjlb/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ab59724a-25f4-4e30-8328-5e32d41a0ca7;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
:: resolution report :: resolve 494ms :: artifacts dl 24ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-a

In [3]:
spark_df = spark.read.parquet("s3a://rwc-ml-datasets/bronze/Flight_Delay_Prediction_Datasets/flight_data/")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

## Type correction

In [8]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql import functions as F

In [10]:
spark_df = spark_df.withColumn(
    "FL_DATE",
    F.to_date(F.to_timestamp(F.col("FL_DATE"), "M/d/yyyy h:mm:ss a"))
)

In [11]:
spark_df.show()

[Stage 2:>                                                          (0 + 1) / 1]

DateTimeException: [CANNOT_PARSE_TIMESTAMP] Text '5/16/2025 12:00:00 AM' could not be parsed, unparsed text found at index 9. Use `try_to_timestamp` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007